# 08 - Hard-Example Mining (Phase 4, redesigned)

**Mode:** GPU (T4 x2 session, GPU 0), background commit, 1.5 h hard cap. Estimated ~0.6 GPU hours.

## Idea
GRPO only learns from prompts whose sampled group has **mixed outcomes** (some pass, some fail). If every sample passes or every sample fails, the group has zero reward variance and contributes nothing. Mining samples each training task n=4 times and classifies it by pass rate:

| class | passes out of 4 | learning signal for GRPO |
|---|---|---|
| solved | 3-4 | little (mostly all-pass groups) |
| borderline | 2 | **high** |
| hard | 1 | **high** |
| impossible | 0 | none (all-fail groups) |

## Why this replaces the local Phase 4 result
Locally, mining found **420/420 impossible**. NB 03 showed that was a prompt bug (the model never saw the function name). With the v2 prompt, the base model solves ~41% of tasks, so mining should now find a real mix. This notebook re-runs it on the 256 training tasks and builds three equal-sized GRPO training sets for an ablation: `hard` (borderline + hard), `random`, `mixed` (70% hard + 30% from solved/impossible).

Dev/eval tasks are never mined (AGENTS.md rule 7).

In [ ]:
INSTALL_WHEELS = False

import glob, os, subprocess, sys, time
T0 = time.time()

def find_input(name):
    hits = [p for p in glob.glob(f"/kaggle/input/**/{name}", recursive=True) if os.path.isdir(p)]
    assert hits, f"dataset '{name}' not attached; found: {os.listdir('/kaggle/input')}"
    return hits[0]

UTILS, DATA = find_input("llm-lab-utils"), find_input("llm-lab-data")
WHEELS = find_input("llm-lab-wheels") if INSTALL_WHEELS else None

# Kaggle's image ships torchao 0.10, which peft>=0.20 refuses to import. We do not use it.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], capture_output=True)

if INSTALL_WHEELS:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--no-deps",
                        *sorted(glob.glob(f"{WHEELS}/*.whl"))], capture_output=True, text=True)
    print("offline pip install rc:", r.returncode, r.stderr[-500:])

if os.path.exists(f"{UTILS}/__init__.py"):
    os.makedirs("/tmp/_pkg", exist_ok=True)
    if not os.path.exists("/tmp/_pkg/llm_lab_utils"):
        os.symlink(UTILS, "/tmp/_pkg/llm_lab_utils")
    sys.path.insert(0, "/tmp/_pkg")
else:
    found = glob.glob(f"{UTILS}/**/llm_lab_utils/__init__.py", recursive=True)
    assert found, "llm_lab_utils package not found in dataset"
    sys.path.insert(0, os.path.dirname(os.path.dirname(found[0])))
import llm_lab_utils
print("llm_lab_utils", llm_lab_utils.__version__, "from", llm_lab_utils.__file__)

In [ ]:
import json, gc, torch, psutil, collections, pandas as pd
from llm_lab_utils import PRIMARY_MODEL_ID, SubprocessVerifier, ExperimentTracker, load_dataset
from llm_lab_utils.model_utils import setup_model_and_tokenizer
from llm_lab_utils.miner import mine_tasks

MODEL_CHOICE = "base"      # <- change after NB 07: base | sft | grpo_from_base | grpo_from_sft
def one(pattern):
    hits = glob.glob(pattern, recursive=True); assert hits, f"not found: {pattern}"; return hits[0]
ADAPTERS = {"base": None,
            "sft": lambda: [one(f"{find_input('04-sft-training')}/**/exp01_sft_mbpp_v1/checkpoint-final")],
            "grpo_from_base": lambda: [one(f"{find_input('06-grpo-training')}/**/grpo_from_base/checkpoint-final")],
            "grpo_from_sft": lambda: [one(f"{find_input('04-sft-training')}/**/exp01_sft_mbpp_v1/checkpoint-final"),
                                      one(f"{find_input('06-grpo-training')}/**/grpo_from_sft/checkpoint-final")]}
adapters = ADAPTERS[MODEL_CHOICE]() if ADAPTERS[MODEL_CHOICE] else None
print("model:", MODEL_CHOICE, adapters)
model, tokenizer = setup_model_and_tokenizer(PRIMARY_MODEL_ID, dtype="float16", device_map={"": 0}, adapter_path=adapters, merge=True)
model.eval()

GEN_CFG = {"max_new_tokens": 384, "temperature": 0.7, "top_p": 0.95, "do_sample": True}     # = local mining_complex.yaml
N_SAMPLES, SEED = 4, 42
tasks = load_dataset(f"{DATA}/mbpp_v2_train.jsonl")
verifier = SubprocessVerifier(timeout=5.0, max_memory_mb=512)
print(len(tasks), "training tasks")

## 1. Mine

In [ ]:
import torch
torch.manual_seed(SEED)
EXP_ID = f"exp03_mining_{MODEL_CHOICE}"
OUT = f"/kaggle/working/{EXP_ID}"
t0 = time.time(); torch.cuda.reset_peak_memory_stats(0)
report = mine_tasks(model, tokenizer, tasks, verifier, N_SAMPLES, GEN_CFG, EXP_ID, OUT,
                    dataset_name="mbpp_v2_train.jsonl", seed=SEED)
wall_h = (time.time() - t0) / 3600
print(report["classification_counts"], f"| {wall_h:.2f} h")

## 2. Analysis: how much of the training set can actually teach GRPO?

In [ ]:
counts = report["classification_counts"]
n = sum(counts.values())
mixed_share = (counts["borderline"] + counts["hard"]) / n
print(pd.Series(counts).to_frame("tasks").assign(share=lambda d: (d.tasks / n).round(3)).to_string())
print(f"\ntasks with a learning signal (borderline+hard): {mixed_share:.1%}")
per_task_c = [t["c_correct"] for t in report["tasks"]]
print("pass-count histogram (0..4):", dict(sorted(collections.Counter(per_task_c).items())))

# does the mined difficulty track reference-solution length? (a cheap sanity check on the classes)
ref_len = {t.id: len(t.reference_solution or "") for t in tasks}
by_cls = collections.defaultdict(list)
for t in report["tasks"]: by_cls[t["category"]].append(ref_len[t["task_id"]])
print(pd.DataFrame({k: {"n": len(v), "mean_ref_chars": round(sum(v) / len(v), 1)} for k, v in by_cls.items()}).T.to_string())
files = {f: sum(1 for _ in open(f"{OUT}/{f}")) for f in ["hard.jsonl", "random.jsonl", "mixed.jsonl"]}
print("\nablation set sizes (should be equal):", files)

## 3. Save

In [ ]:
tracker = ExperimentTracker(EXP_ID + "_summary", "/kaggle/working")
tracker.set_config(model=f"{PRIMARY_MODEL_ID} ({MODEL_CHOICE}: {adapters})", dataset="mbpp_v2_train.jsonl",
                   hyperparameters={**GEN_CFG, "n_samples": N_SAMPLES, "seed": SEED, "dtype": "float16", "merged": True})
wall_total = (time.time() - T0) / 3600
tracker.update_metrics({"counts": counts, "mixed_share": mixed_share, "ablation_sizes": files})
tracker.update_hardware(torch.cuda.max_memory_allocated(0) / 1e9, psutil.virtual_memory().used / 1e9)
tracker.data["training"].update(wall_hours=round(wall_total, 4), gpu_hours=round(wall_total, 4)); tracker.save()
print(f"saved. wall = GPU hours = {wall_total:.2f}")

## Takeaways (fill in)
- What share of training tasks has a learning signal? If it is small, GRPO on all 256 tasks wastes most of its steps; if large, the 0.25-0.35 train reward in NB 06 was mostly diversity limited, not task limited.
- Next: a GRPO ablation on `hard.jsonl` vs `random.jsonl` vs `mixed.jsonl` (same size, same hyperparameters) answers whether mining pays for itself in GPU-hours.